# Functions

One of the greatest ability of the human mind is the ability to abstract - and what could be a better concept of abstraction then the concept of functions? Functions help us compact infinite information into a finite space. 

For example, consider the very loved function, $F(x) = x^2$. If we consider only positive integer values for x, the information contained in this function is equivalent to the following table:

| Value of x | Value of F(x) |
| ---------- | ------------- |
| 1 | 1 |
| 2 | 4 |
| 3 | 9 |
| 4 | 16 | 
| ... | ... |

... we can keep filling this table all the way to $+\infty$. This function, **F(x)**, essentially converts this infinite length table in a simple one line statement: $F(x) = x^2$.

This is is the power of functons. Functions are able to abstract out a very large (or even $+\infty$) **state spaces** to a finite and consise form. 

This gives us the motivation to develop function-based Learning Methods where we replace the **Q Table** (which is a table with state-action pairs) to a **Q Function** - which is a function that aims to compact the Table of Q into a simple and concise statement form.

## Baby Steps

First lets consider our favourite, Monte Carlo Update Algorithm!

The Update equation is:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big( G_t - Q(s, a) \Big)$$


Where Q(s,a) is a Q Table which may like:

| State s | Action a | Value of Q(s, a) |
| ---------- | ------------- | --- |
| 1 | 1 | 0.4 | 
| 1 | 2 | - 0.2 | 
| 1 | 3 | 4 |
| 1 | 4 | 9 |
| 2 | 1 | 67 |
| 2 | 2 | 2 |
| 2 | 3 | 4 |
| 2 | 4 |  5 |
| ... | ... | ...|

Obviously the values of Q(s,a) is arbitrary in this table. But the size of this table will be | S | * | A | where S is the set of states and A is the Set of Actions. For cliff walking, this table will have a size of ( 37 ) * ( 4 ) = 148 distinct rows.

Imgine, instead of having 37 states, we had 10,000 states and 200 actions. We would have 2,000,000 distinct rows whose Q(s, a) we will have to "learn" by interacting with the environment. Firstly, that would take a long time to update accurate values, and, in some cases might even be impossible to learn in one human lifetime (80 years). Another issue is the issue of constantly keeping this much information in memory - 2,000,000 distict rows is no joke to store.

This is where we can use functions to *approximate* the values of Q(s, a) instead of storing the entire table, we will only store the function and calculate the actual Q-Values in runtime!

So, consider the following Q Function:

$$Q_{function} (w, a, s) = F(w, a, s) $$

Where w is a vector (which is a learnable parameter), and assume F(w, a, s) gives us the correct value of taking action, a, at state s for any valid values of a and s.

The our policy can be rewritten from:

$$
\pi_{*} = \argmax_{a} (Q_{table}(a, s));


\forall a \in A, s \in S
$$

to

$$
\pi_{*} = \argmax_{a} (Q_{function}(w, a, s));
\forall a \in A, s \in S
$$ 

when Q-Table and $Q_{function} are optimal.

However, our problem then becomes how can I update the value of w s.t F(w, a, s) will always output the correct Q(s, a) Value. Moreover, how can we decide how F(w, a, s) will look like?



### How does F(w, a, s) look like?

The straightforward answer is - *we don't know*. There is probably no universal function, F(w, a, s), that will work for solving all problems. But we can take guess a function that can work and check if it is able to correctly estimate values of Q(s, a). Lets look at some candidate functions for our table:

#### Linear Function

$F(w, a, s) = w_{1} * s + w_{2} * s + w_{3} * s + w_{4} * s + a$

Where $w_k$ is the $k-th$ element of vector $w$

For a 4 dimentional vector w. Of course, we can use more dimentions for vector w and see if it can work. 

---

#### Power Function
$ F(w, a, s) = w_{1} * s + w_{2} * s^2 + w_{3} * s^3 + w_{4} * s^4 + a$

For a 4 dimentional vector w. Of course, we can use more dimentions for vector w and see if it can work.

---


#### Core Properties

The core properties for the candidate function F(w, a, s) is that it can:

1. Domain of s, is greater than or requal to the set, S, which contains all values of s. (Trivial)
2. Domain of a, is greater than or requal to the set, A, which contains all values of a. (Trivial)
3. Correctly able to map state-action space to value space (Difficult)

The third point is the one that is diffult to satisfy. We have to perform trial and error to really know which function works.


## Optimise F

Once we have a candidate Fuction, we need to optimise w. We want to reduce the error:

$$ min_{w}(G_t - F(w, s, a)) $$

Where $G_t$ is the actual return we get at state s, taking action a and following the our policy $\pi$ until termination. However, instead of minsing $ (G_t - F(w, s, a)) $, we should mininmise:

$$ min_{w} [G_t - F(w, s, a)]^2 $$

Because it will penalise greater deviations more. Furthermore, we are not trying to minimise our Error at just state s and a, but for all states and all actions. We can write all that down succintly by saying:

$$ min_{w} \sum_{a , s} [G_t - F(w, s, a)]^2 ; \forall a \in A, s \in S  $$


### Minimise

Now let's actually perform minization. Easy way to do this, is simply differentiating F and then following **down-slope**, of course, given F is differentiable. Which would mean:

$$ w \leftarrow w - \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$

Note that if we want to go above slope, we would use a **plus** (+) sign instead of **minus** (-)

Now lets incorporate a learning rate, call it $\alpha$ and introduce a constant scaling of $\frac{1}{2}$. Our equation will become:

$$w \leftarrow w - \alpha \frac{1}{2} \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$

Lets simplify by performing the differntiation:

$$   \alpha \frac{1}{2} \frac{\partial}{\partial w} \sum_{a , s} [G_t - F(w, s, a)]^2 $$
$$ =  \alpha \frac{1}{2} \sum_{a , s} 2 * [G_t - F(w, s, a)] * (-1) \frac{\partial}{\partial w} F(w, s, a) $$
$$ = \alpha \frac{1}{2} * 2 * (-1) \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$
$$ =  - \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$


Substituting the above differentiation to our optimization equation, it becomes:

$$ w \leftarrow w - ( - \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)] )$$

or, simply

$$ w \leftarrow w + \alpha \sum_{a , s} [[G_t - F(w, s, a)] * \frac{\partial}{\partial w} F(w, s, a)]$$



### Limitations

Notice that we now have limitations with $ F(w, s, a)$ since we have to calculate $\frac{\partial}{\partial w} F(w, s, a)$ which means our q-function, F, has to be differentiable. Another limitation is that we need the complete return, $G_t$, of an episode before we start making updates to our q-function. So our Agent will be acting completely randomly on the first episode and if the episodes are very long, our agent will take a long time to converge to optimal policy. 

Keeping these limitations in mind, let's see an example on how we can implement a functional approach to RL instead of the traditional q tables.

### Implementation

Instead of worrying about whether our function will be able to effectively model the environment, lets use a fool proof function to do it! Note that for cliff walking there are less than 48 states (4*12 = 48).

Define:
$$ F(w,s,a) = w_a * x(s) $$

s.t 

$$ \frac{\partial}{\partial w} F(w,s,a) =  x(s) $$

Where:
- $w_a$ is a 48-dimensional vector depending on value of a. so if we have $w_0$, $w_1$, $w_2$ and $w_3$
- x(s) is a 48-dimensional vector that depends on s. We define x(s) as:

$$ x_k = \begin{cases} \text{1} &  \text{if k = s } \\ 
\text{0} &  \text{if k } {\neq s } \end{cases} 
\quad \text{; where } x_k \text{ is the } k^{th} \text{term in } x(s) 
$$

in code it may look like:

```python

def x(s):
    temp = np.zeros(48) # 48 columns
    temp[s] = 1
    return temp

w = np.zeros((4, 48)) # 48 cols, 4 rows

def F(w, s, a):
    weights = w[a]
    result = np.dot(weights, x(s))

    return result


def dwF(w, s, a):
    return x(s)
```

This is a very popular candidate Q-Function and is known as Linear Value Function Approximation with one-hot state encoding.

### Implementation

Lets actually implement this in python!

In [ ]:
from utils import *
import numpy as np
import gymnasium as gym

epsilon = 0.1
epsilon_decay_rate = 0.999
epsilon_min = 0.01

gemma = 0.9
alpha = 0.1
num_episodes = 5000
MAX_STEPS_PER_EPISODE = 500

seed = 42

env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)

weights = np.zeros((len(action_space), len(observation_space)))

def x(state):
    temp = np.zeros(len(observation_space))
    temp[state] = 1
    return temp

def q_function(weights, state, action):
    weight_vector = weights[action]
    value = np.dot(weight_vector, x(state=state))
    return value

def q_function_differential(weights, state, action):
    return x(state=state)


episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []

In [ ]:
rng = np.random.default_rng(seed=seed)


def get_best_acton(weights, state):
    num_actions = weights.shape[0]
    best_action_idx = 0
    best_action_val = weights[0, state]
    for action in range(num_actions):
        if best_action_val < weights[action, state]:
            best_action_idx = action
            best_action_val = weights[action, state]

    return best_action_idx


for episode in range(num_episodes):

    start_time = datetime.now()
    env.reset()

    # Array to store, State, Action, Reward for the current episode. WE use it to backtrack and update the Q-values at the end of the episode.
    episode_data = []
    terminated = False
    truncated = False

    state, info = env.reset()
    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)  # Decay epsilon after each episode

    while not (terminated or truncated):

        # Epsilon-greedy action selection
        if rng.random() < epsilon:
            action = rng.choice(action_space)
        else:
            action = get_best_acton(weights=weights, state=state)

        next_state, reward, terminated, truncated, info = env.step(action)

        episode_data.append((state, action, reward))

        state = next_state

    # Calculate returns and update Q-Function weights
    G = 0
    for state, action, reward in reversed(episode_data):
        G = gemma * G + reward
        TD_Error = G - q_function(weights=weights, state=state, action=action)
        weights[action] += alpha * TD_Error * q_function_differential(weights=weights, state=state, action=action)


    # Average episode error for monitoring convergence
    Avg_MC_Episode_Error = np.mean([np.abs(G - q_function(weights=weights, state=state, action=action)) for state, action, reward in episode_data])
    episode_avg_mc_error.append(Avg_MC_Episode_Error)


    end_time = datetime.now()
    episode_duration = (end_time - start_time).total_seconds()
    episode_durations.append(episode_duration)
    episode_rewards.append(sum([reward for _, _, reward in episode_data]))
    episode_lengths.append(len(episode_data))


    if (episode + 1) % 200 == 0:
        print(f"Episode: {episode + 1}, Average MC Episode Error: {Avg_MC_Episode_Error:.4f}",
                f"Episode Duration: {episode_duration:.4f} seconds",
                f"Total Reward: {episode_rewards[-1]}, Episode Length: {episode_lengths[-1]}",
                f"Epsilon: {epsilon:.4f}")

### Neural Networks


In [ ]:
import torch
from torch import nn, optim
from utils import *
import numpy as np
import gymnasium as gym
import os

epsilon = 0.5
epsilon_decay_rate = 0.9999
epsilon_min = 0.01

gemma = 0.9
alpha = 0.001
num_episodes = 5000
MAX_STEPS_PER_EPISODE = 500

seed = 42
rng = np.random.default_rng(seed=seed)

env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)


input_dim = len(observation_space)
output_dim = len(action_space)

device = torch.device("cpu")

print(f"Using device: {device}")

class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.liner_relu_stacks = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.Linear(64, output_dim)        
            )
    
    def forward(self, x):
        return self.liner_relu_stacks(x)
        

qNetwork = QNetwork(input_dim=input_dim, output_dim=output_dim)
qNetwork = qNetwork.to(device)
optimizer = optim.SGD(qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

state, info = env.reset()

episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []


weights_path = "q_network_weights.pth"
training_data_path = "train_data.json"
isDelete = False
## Delete weights and data
if os.path.exists(weights_path) and isDelete:
    os.remove(weights_path)
    print(f"🗑️ Weight file '{weights_path}' has been deleted from your disk.")

if os.path.exists(training_data_path) and isDelete:
    os.remove(training_data_path)
    print(f"🗑️ Weight file '{training_data_path}' has been deleted from your disk.")


def get_state_tensor(state, observation_space):
        one_hot = np.zeros(len(observation_space))
        one_hot[state] = 1
        state_tensor = torch.tensor(one_hot).unsqueeze(0).float().to(device)
        return state_tensor


def get_best_acton(qNetwork, state, observation_space):
    with torch.no_grad():
        state_input = get_state_tensor(state, observation_space=observation_space)
        logits = qNetwork(state_input)
        return torch.argmax(logits).item()

def choose_action(state, action_space, observation_space, epsilon):
    if rng.random() < epsilon:
        action = rng.choice(action_space)
    else:
        action = get_best_acton(state=state, qNetwork=qNetwork, observation_space=observation_space)
    return action


def generate_qtable(qNetwork, observation_space):
    q_table = np.zeros((len(observation_space), len(action_space)))

    idx = 0
    for action_values in q_table:
        with torch.no_grad():
                state_tensor = get_state_tensor(state=idx, observation_space=observation_space)
                predicted_vals = qNetwork(state_tensor)
                q_table[idx] = np.array(predicted_vals[0])
        idx+=1
    return q_table
    

In [ ]:

if os.path.exists(weights_path):
    qNetwork.load_state_dict(torch.load(weights_path))
    q_table = generate_qtable(qNetwork, observation_space=observation_space)
    plot_q_table(q_table=q_table)

for episode in range(num_episodes):
        
    start_time = datetime.now()
    env.reset()

    # Array to store, State, Action, Reward for the current episode. WE use it to backtrack and update the Q-values at the end of the episode.
    episode_data = []
    terminated = False
    truncated = False

    state, info = env.reset()
    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)  # Decay epsilon after each episode

    while not (terminated or truncated):

        # Epsilon-greedy action selection
        action = choose_action(state=state, action_space=action_space, epsilon=epsilon, observation_space=observation_space)

        next_state, reward, terminated, truncated, info = env.step(action)
        episode_data.append((state, action, reward))
        state = next_state

    # Calculate returns and update Q-Function weights
    G = 0
    absolute_errors = []

    optimizer.zero_grad()

    for state, action, reward in reversed(episode_data):
        G = gemma * G + reward

        state_tensor = get_state_tensor(state=state, observation_space=observation_space)

        q_values = qNetwork(state_tensor)

        predicted_q = q_values[0, action]

        target_tensor = torch.tensor(G, dtype=torch.float32).to(device)
        loss = loss_fn(predicted_q, target_tensor)

        with torch.no_grad():
            absolute_errors.append(np.abs(G - predicted_q.item()))


        torch.nn.utils.clip_grad_norm_(qNetwork.parameters(), max_norm=1.0)
        loss.backward()

    optimizer.step()


    # Debug check
    if torch.isnan(q_values).any():
        print(f"CRITICAL: NaN detected at episode {episode}!")
        print(f"Current Q-values: {q_values}")
        break # Stop training immediately so you can inspect the data

    # Average episode error for monitoring convergence
    Avg_Episode_Error = np.mean(absolute_errors)
    episode_avg_mc_error.append(Avg_Episode_Error)

    end_time = datetime.now()
    episode_duration = (end_time - start_time).total_seconds()
    episode_durations.append(episode_duration)
    episode_rewards.append(sum([reward for _, _, reward in episode_data]))
    episode_lengths.append(len(episode_data))



    if (episode + 1) % 100 == 0:
        print(f"Episode: {episode + 1}, Average MC Episode Error: {Avg_Episode_Error:.4f}",
                f"Episode Duration: {episode_duration:.4f} seconds",
                f"Total Reward: {episode_rewards[-1]}, Episode Length: {episode_lengths[-1]}",
                f"Epsilon: {epsilon:.4f}")
        torch.save(qNetwork.state_dict(), weights_path)

        q_table = generate_qtable(qNetwork, observation_space=observation_space)
        plot_q_table(q_table=q_table)


## Q-Learning with Deep Networks

In [ ]:
import torch
from torch import nn, optim
from utils import *
import numpy as np
import gymnasium as gym
import os

epsilon = 0.5
epsilon_decay_rate = 0.9999
epsilon_min = 0.01

gamma = 0.9
alpha = 0.001
num_episodes = 5000
MAX_STEPS_PER_EPISODE = 500

seed = 42
rng = np.random.default_rng(seed=seed)

env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)


input_dim = len(observation_space)
output_dim = len(action_space)

device = torch.device("cpu")

print(f"Using device: {device}")

class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.liner_relu_stacks = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.Linear(64, output_dim)        
            )
    
    def forward(self, x):
        return self.liner_relu_stacks(x)
        

qNetwork = QNetwork(input_dim=input_dim, output_dim=output_dim)
qNetwork = qNetwork.to(device)
optimizer = optim.SGD(qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

state, info = env.reset()

episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []


weights_path = "q_network_weights.pth"
training_data_path = "train_data.json"
isDelete = False
## Delete weights and data
if os.path.exists(weights_path) and isDelete:
    os.remove(weights_path)
    print(f"🗑️ Weight file '{weights_path}' has been deleted from your disk.")

if os.path.exists(training_data_path) and isDelete:
    os.remove(training_data_path)
    print(f"🗑️ Weight file '{training_data_path}' has been deleted from your disk.")


def get_state_tensor(state, observation_space):
        one_hot = np.zeros(len(observation_space))
        one_hot[state] = 1
        state_tensor = torch.tensor(one_hot).unsqueeze(0).float().to(device)
        return state_tensor


def get_best_acton(qNetwork, state, observation_space):
    with torch.no_grad():
        state_input = get_state_tensor(state, observation_space=observation_space)
        logits = qNetwork(state_input)
        return torch.argmax(logits).item()

def choose_action(state, action_space, observation_space, epsilon):
    if rng.random() < epsilon:
        action = rng.choice(action_space)
    else:
        action = get_best_acton(state=state, qNetwork=qNetwork, observation_space=observation_space)
    return action


    

In [ ]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import deque
from datetime import datetime

# --- Hyperparameters ---
gamma = 0.9
epsilon = 0.5
epsilon_decay_rate = 0.9995  # Slower decay for neural network stability
epsilon_min = 0.05
alpha = 0.001                # Controlled learning rate for Adam
num_episodes = 2000
batch_size = 64              # Experience batch size
target_update_frequency = 10 # Sync after 10 episodes

device = torch.device("cpu") # Use CPU for small grid latency efficiency


# ------- Setup -----
MAX_STEPS_PER_EPISODE = 500

seed = 42
rng = np.random.default_rng(seed=seed)
env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)


# --- Replay Buffer Initialization ---
replay_buffer = deque(maxlen=20000) # Holds past transitions

# --- Model setup (Using LeakyReLU to prevent dead neurons) ---
class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(),
            nn.Linear(64, 64),
            nn.LeakyReLU(),
            nn.Linear(64, output_dim)        
        )
    def forward(self, x):
        return self.network(x)

# Double networks to decouple target calculation from active tracking
policy_qNetwork = QNetwork(len(observation_space), len(action_space)).to(device)
target_qNetwork = QNetwork(len(observation_space), len(action_space)).to(device)

target_qNetwork.load_state_dict(policy_qNetwork.state_dict()) # Sync initially

optimizer = optim.Adam(policy_qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

# --- Helper Functions ---
def get_state_tensor(state, observation_space):
    one_hot = np.zeros(len(observation_space))
    one_hot[state] = 1
    return torch.tensor(one_hot).unsqueeze(0).float().to(device)

def choose_action(state, epsilon):
    if rng.random() < epsilon:
        return rng.choice(action_space)
    else:
        with torch.no_grad():
            state_tensor = get_state_tensor(state, observation_space)
            q_values = policy_qNetwork(state_tensor)
            return torch.argmax(q_values).item()
        
def generate_qtable(qNetwork, observation_space):
    q_table = np.zeros((len(observation_space), len(action_space)))

    idx = 0
    for action_values in q_table:
        with torch.no_grad():
                state_tensor = get_state_tensor(state=idx, observation_space=observation_space)
                predicted_vals = qNetwork(state_tensor)
                q_table[idx] = np.array(predicted_vals[0])
        idx+=1
    return q_table

def load_DQN_weights():
    weights_path = "DQN_weights.pth"
    if os.path.exists(weights_path):
        policy_qNetwork.load_state_dict(torch.load(weights_path))
        target_qNetwork.load_state_dict(torch.load(weights_path))

def save_DQN_weights():
    weights_path = "DQN_weights.pth"
    torch.save(policy_qNetwork.state_dict(), weights_path)


load_DQN_weights()
q_table = generate_qtable(qNetwork=policy_qNetwork, observation_space=observation_space)
plot_q_table(q_table)

# --- DQN Main Loop ---
for episode in range(num_episodes):
    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False
    episode_reward = 0
    start_time = datetime.now()

    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)
    td_errors_in_episode = []

    while not (terminated or truncated):
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward

        # 1. Store experience inside our replay buffer
        done = terminated or truncated
        replay_buffer.append((state, action, reward, next_state, done))
        state = next_state

        # 2. Train only if we have enough data collected in the buffer
        if len(replay_buffer) > batch_size:
            # Sample a random uncorrelated mini-batch
            mini_batch = random.sample(replay_buffer, batch_size)
            
            states, actions, rewards, next_states, dones = zip(*mini_batch)

            # Process entire batches simultaneously to leverage matrix parallelism
            state_tensors = torch.cat([get_state_tensor(s, observation_space) for s in states])
            next_state_tensors = torch.cat([get_state_tensor(ns, observation_space) for ns in next_states])
            
            reward_tensors = torch.tensor(rewards, dtype=torch.float32).to(device)
            action_tensors = torch.tensor(actions, dtype=torch.long).to(device)
            done_tensors = torch.tensor(dones, dtype=torch.float32).to(device)

            # Get current Q predictions for the chosen actions
            current_q_values = policy_qNetwork(state_tensors)
            predicted_q = current_q_values.gather(1, action_tensors.unsqueeze(1)).squeeze(1)

            # Compute TD Target using the frozen TARGET network
            with torch.no_grad():
                max_next_q = target_qNetwork(next_state_tensors).max(1)[0]
                td_target = reward_tensors + (gamma * max_next_q * (1 - done_tensors))

            # Optimize weights via MSE loss minimization
            loss = loss_fn(predicted_q, td_target)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_qNetwork.parameters(), max_norm=1.0) # Safety clip
            optimizer.step()

            with torch.no_grad():
                td_errors_in_episode.append(torch.abs(td_target - predicted_q).mean().item())

    # 3. Periodically sync Target network parameters with Policy network
    if (episode + 1) % target_update_frequency == 0:
        target_qNetwork.load_state_dict(policy_qNetwork.state_dict())

    # Metrics Logging
    end_time = datetime.now()
    avg_td_error = np.mean(td_errors_in_episode) if td_errors_in_episode else 0.0
    if (episode + 1) % 100 == 0:
        q_table = generate_qtable(qNetwork=policy_qNetwork, observation_space=observation_space)
        plot_q_table(q_table)
        print(f"Episode {episode + 1}/{num_episodes} - Total Reward: {episode_reward}, Epsilon: {epsilon:.3f}, Avg TD Error: {avg_td_error:.4f}")
        save_DQN_weights()

## MOnte Carlo Retry

In [ ]:
import os
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from datetime import datetime
from utils import *

# --- Hyperparameters ---
gamma = 0.9
epsilon = 0.5
epsilon_decay_rate = 0.9995  # Slow decay for neural network stability
epsilon_min = 0.05
alpha = 0.001                # Stable learning rate for Adam
num_episodes = 2000
weights_path = "monteCarlo_qNetwork.pth"

device = torch.device("cpu") # Kept on CPU to prevent M4 kernel launch overhead

# ------- Setup -----
MAX_STEPS_PER_EPISODE = 500

seed = 42
rng = np.random.default_rng(seed=seed)
env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)

action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)


# --- Model setup (Using LeakyReLU to prevent dead neurons) ---
# class QNetwork(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super(QNetwork, self).__init__()
#         self.network = nn.Sequential(
#             nn.Linear(input_dim, 64),
#             nn.LeakyReLU(),
#             nn.Linear(64, 64),
#             nn.LeakyReLU(),
#             nn.Linear(64, output_dim)        
#         )
#     def forward(self, x):
#         return self.network(x)
    
class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.liner_relu_stacks = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(),
            nn.Linear(64, output_dim)        
            )
    
    def forward(self, x):
        return self.liner_relu_stacks(x)
        

# Initialize network and optimizer
qNetwork = QNetwork(len(observation_space), len(action_space)).to(device)
optimizer = optim.Adam(qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

# ==========================================
# WEIGHTS HELPER FUNCTIONS
# ==========================================
def load_checkpoint(model, optimizer, file_path):
    if os.path.exists(file_path):
        print(f"🔄 Found existing checkpoint '{file_path}'. Loading weights...")
        checkpoint = torch.load(file_path)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        return checkpoint.get('epsilon', 1.0), checkpoint.get('episode', 0)
    else:
        print(f"⚠️ No checkpoint found at '{file_path}'. Starting from scratch.")
        return 1.0, 0

def save_checkpoint(model, optimizer, epsilon, episode, file_path):
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epsilon': epsilon,
        'episode': episode
    }, file_path)

# Execute load before training loop begins
start_episode = 0
# epsilon, start_episode = load_checkpoint(qNetwork, optimizer, weights_path)


# --- Helper Functions ---
def get_state_tensor(state, observation_space):
    one_hot = np.zeros(len(observation_space))
    one_hot[state] = 1
    return torch.tensor(one_hot).unsqueeze(0).float().to(device)

def choose_action(state, epsilon):
    if rng.random() < epsilon:
        return rng.choice(action_space)
    else:
        with torch.no_grad():
            state_tensor = get_state_tensor(state, observation_space)
            q_values = qNetwork(state_tensor)
            return torch.argmax(q_values).item()
        
def generate_qtable(qNetwork, observation_space):
    q_table = np.zeros((len(observation_space), len(action_space)))

    for idx in range(len(q_table)):
        with torch.no_grad():
                state_tensor = get_state_tensor(state=idx, observation_space=observation_space)
                predicted_vals = qNetwork(state_tensor)
                q_table[idx] = np.array(predicted_vals[0], copy=True, dtype=float)
    return q_table

# ==========================================
# View Initial Q Table
# ==========================================

q_table = generate_qtable(qNetwork=qNetwork, observation_space=observation_space)
plot_q_table(q_table)

# ==========================================
# MAIN MONTE CARLO LOOP
# ==========================================
for episode in range(start_episode, num_episodes):
    state, info = env.reset(seed=seed)
    terminated = False
    truncated = False
    episode_reward = 0
    episode_data = [] # Stores (state, action, reward)
    start_time = datetime.now()

    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)

    # 1. Generate an entire episode trajectory
    while not (terminated or truncated):
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        
        episode_data.append((state, action, reward))
        state = next_state

    # 2. Backtrack to compute MC returns and accumulate gradients
    G = 0
    mc_errors_in_episode = []
    
    # Reset gradients ONCE before processing the episode data batch
    optimizer.zero_grad()
    
    for state, action, reward in reversed(episode_data):
        G = gamma * G + reward  # Total discounted return from time t onwards

        state_tensor = get_state_tensor(state, observation_space)
        q_values = qNetwork(state_tensor)
        predicted_q = q_values[0, action]

        target_tensor = torch.tensor([G], dtype=torch.float32).to(device)
        
        # Standard MSE Loss
        loss = loss_fn(predicted_q.unsqueeze(0), target_tensor)
        with torch.no_grad():
            margin = 0.05
            assert (G - predicted_q.item()) ** 2 * (1 - margin) <= loss.item() <= (G - predicted_q.item()) ** 2 * (1 + margin), f"Assert Correct Loss Val! Actual Loss {(G - predicted_q.item()) ** 2 }, Calculated Loss {loss.item()}"

        # Accumulate gradients (adds them together without updating weights yet)
        loss.backward()

        with torch.no_grad():
            mc_errors_in_episode.append(torch.abs(target_tensor - predicted_q).item())

    # 3. Apply Gradient Clipping and step the optimizer ONCE per episode
    if len(episode_data) > 0:
        torch.nn.utils.clip_grad_norm_(qNetwork.parameters(), max_norm=1.0)
        optimizer.step()

    # Metrics and Logging
    end_time = datetime.now()
    avg_mc_error = np.mean(mc_errors_in_episode) if mc_errors_in_episode else 0.0
    
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}/{num_episodes} - Total Reward: {episode_reward}, Epsilon: {epsilon:.3f}, Avg MC Error: {avg_mc_error:.4f}")
        # Periodic auto-save every 100 episodes
        q_table = generate_qtable(qNetwork=qNetwork, observation_space=observation_space)
        plot_q_table(q_table)
        save_checkpoint(qNetwork, optimizer, epsilon, episode, weights_path)

# Final save when training finishes completely
save_checkpoint(qNetwork, optimizer, epsilon, num_episodes, weights_path)

In [ ]:
import torch
from torch import nn, optim
from utils import *
import numpy as np
import gymnasium as gym
import os
import csv
import io

import warnings
warnings.filterwarnings('ignore')


gamma = 0.9
epsilon = 0.5
epsilon_decay_rate = 0.9995  # Slow decay for neural network stability
epsilon_min = 0.05
alpha = 0.001                # Stable learning rate for Adam
num_episodes = 2000


seed = 42
rng = np.random.default_rng(seed=seed)
MAX_STEPS_PER_EPISODE = 500
env = gym.make("CliffWalking-v1", max_episode_steps=MAX_STEPS_PER_EPISODE)
action_space = np.arange(env.action_space.n)
observation_space = np.arange(env.observation_space.n)

device = torch.device("cpu")

print(f"Using device: {device}")

class QNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(QNetwork, self).__init__()

        self.liner_relu_stacks = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.LeakyReLU(),
            nn.Linear(64, output_dim)        
            )
    
    def forward(self, x):
        return self.liner_relu_stacks(x)
        
qNetwork = QNetwork(len(observation_space), len(action_space)).to(device)
optimizer = optim.Adam(qNetwork.parameters(), lr=alpha)
loss_fn = nn.MSELoss()

state, info = env.reset()

episode_lengths = []
episode_rewards = []
episode_durations = []
episode_avg_mc_error = []
states_visited = {}

for i in range(len(observation_space)):
    states_visited[i] = 0


weights_path = "q_network_weights.pth"
training_data_path = "train_data.json"


def delete_weights():
    ## Delete weights and data
    if os.path.exists(weights_path):
        os.remove(weights_path)
        print(f"🗑️ Weight file '{weights_path}' has been deleted from your disk.")

def get_state_tensor(state, observation_space):
    one_hot = np.zeros(len(observation_space))
    one_hot[state] = 1
    return torch.tensor(one_hot).unsqueeze(0).float().to(device)

def choose_action(state, epsilon):
    if rng.random() < epsilon:
        return rng.choice(action_space)
    else:
        with torch.no_grad():
            state_tensor = get_state_tensor(state, observation_space)
            q_values = qNetwork(state_tensor)
            return torch.argmax(q_values).item()
        
def generate_qtable(qNetwork, observation_space):
    q_table = np.zeros((len(observation_space), len(action_space)))

    for idx in range(len(q_table)):
        with torch.no_grad():
                state_tensor = get_state_tensor(state=idx, observation_space=observation_space)
                predicted_vals = qNetwork(state_tensor)
                q_table[idx] = np.array(predicted_vals[0], copy=True, dtype=float)
    return q_table

def print_state_statistics(states_visited_dict, output_filename="state_visit_statistics.csv"):
    sorted_states = sorted(states_visited_dict.items(), key=lambda item: item[1], reverse=True)
    
    # 2. Print out the ordered statistics to the console
    print("\n📊 --- State Visit Statistics (Ordered) ---")
    print(f"{'State ID':<10} | {'Visits':<10}")
    print("-" * 25)
    for state, visits in sorted_states:
        if state == 47:
            print(f"{state:<10} (Goal State) | {visits:<10}")
        elif state == 37:
            print(f"{state:<10} (Last State) | {visits:<10}")
        else:
            print(f"{state:<10} | {visits:<10}")
        
    with open(output_filename, mode="w", newline="", encoding="utf-8") as f:
        csv_writer = csv.writer(f)
        csv_writer.writerow(["State_ID", "Visit_Count"])
        csv_writer.writerows(sorted_states)
        
    print(f"\n💾 CSV file successfully saved to disk as: '{output_filename}'")


q_table = generate_qtable(qNetwork=qNetwork, observation_space=observation_space)
plot_q_table(q_table=q_table)

for episode in range(num_episodes):
        
    start_time = datetime.now()
    env.reset()

    # Array to store, State, Action, Reward for the current episode. WE use it to backtrack and update the Q-values at the end of the episode.
    episode_data = []
    episode_reward = 0
    terminated = False
    truncated = False

    state, info = env.reset()
    epsilon = max(epsilon * epsilon_decay_rate, epsilon_min)  # Decay epsilon after each episode


    while not (terminated or truncated):
        action = choose_action(state, epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        episode_reward += reward
        
        episode_data.append((state, action, reward))
        state = next_state
        

    # Calculate returns and update Q-Function weights
    G = 0
    absolute_errors = []
    optimizer.zero_grad()
    for state, action, reward in reversed(episode_data):
        G = reward + gamma * G 

        state_tensor = get_state_tensor(state=state, observation_space=observation_space)
        states_visited[state] += 1
        assert state_tensor[0][state].item() == 1, "Assert correct state tensor"

        q_values = qNetwork(state_tensor)
        
        predicted_q = q_values[0, action]
        target_tensor = torch.tensor([G], dtype=torch.float32).to(device)
        
        loss = loss_fn(predicted_q.unsqueeze(0), target_tensor)
        with torch.no_grad():
            expected_loss_val = (G - predicted_q.item()) ** 2
            expected_tensor = torch.tensor([expected_loss_val], dtype=torch.float32).to(device)
            
            assert torch.allclose(loss, expected_tensor, atol=1e-5), \
                f"Loss mismatch! Manual: {expected_loss_val:.6f}, PyTorch: {loss.item():.6f}"
                    

        loss.backward()

        torch.nn.utils.clip_grad_norm_(qNetwork.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()
        
        with torch.no_grad():
            absolute_errors.append(torch.abs(target_tensor - predicted_q).item())

  


    # Average episode error for monitoring convergence
    Avg_Episode_Error = np.mean(absolute_errors)
    episode_avg_mc_error.append(Avg_Episode_Error)

    end_time = datetime.now()
    episode_duration = (end_time - start_time).total_seconds()
    episode_durations.append(episode_duration)
    episode_rewards.append(sum([reward for _, _, reward in episode_data]))
    episode_lengths.append(len(episode_data))

    if (episode + 1) % 100 == 0:
        print(f"Episode: {episode + 1}, Average MC Episode Error: {Avg_Episode_Error:.4f}",
                f"Episode Duration: {episode_duration:.4f} seconds",
                f"Total Reward: {episode_rewards[-1]}, Episode Length: {episode_lengths[-1]}",
                f"Epsilon: {epsilon:.4f}")
        torch.save(qNetwork.state_dict(), weights_path)
        q_table = generate_qtable(qNetwork, observation_space=observation_space)
        plot_q_table(q_table=q_table)




#### Comments

`SGD Optimizer` seems to descend to the optimum too slowly - requiring far more than 2000 episodes and slower epsilon decay (or a different epsilon strategy entirely). However, `Adam Optimizer` with the power of varying learning rates for higher slopes learn much faster and weights descents to local optima in less than 2000 episodes.

In [ ]:
plot_q_table(q_table=q_table)
print_state_statistics(states_visited_dict=states_visited)